# Beat Tracker and Tempo Estimation
- Asa Picton, Connor Richardson, Dylan Baecker
- CSC475

### 1. Import libraries

In [7]:

from pathlib import Path
from joblib import Parallel, delayed
import joblib
import numpy as np
import librosa as lb
from tqdm.auto import tqdm


### 2. Load audio and tempo data

In [8]:
audio_path = Path("giantsteps-tempo-dataset/audio")
label_path = Path("giantsteps-tempo-dataset/annotations/tempo")

# Load tempo labels
tempo_labels = {}
for file in label_path.iterdir():
    tempo_labels[file.stem] = float(file.read_text().strip())

# Build dataset
dataset = []
for file in audio_path.iterdir():
    track_id = file.stem
    if track_id in tempo_labels:
        dataset.append((file, tempo_labels[track_id]))

print(f"Loaded {len(dataset)} audio tracks with tempo labels")


Loaded 664 audio tracks with tempo labels


In [3]:
print(len(dataset))
print(dataset[0])
print(dataset[0][0].exists())

664
(PosixPath('giantsteps-tempo-dataset/audio/3226172.LOFI.mp3'), 69.0)
True


### 3. Feature Extraction

In [ ]:
def root_mean_square(y_audio, sr):
    rms_vals = lb.feature.rms(y=y_audio)[0]
    rms_features = np.array([
        np.mean(rms_vals),
        np.std(rms_vals),
        np.max(rms_vals),
        np.min(rms_vals)
    ])
    return rms_features

def onset_envelope(y_audio, sr):
    onset_env = lb.onset.onset_strength(y=y_audio, sr=sr)
    onset_features = np.array([
        np.mean(onset_env),
        np.std(onset_env),
        np.max(onset_env),
            np.min(onset_env)
    ])
    return onset_features


def process_track(path, bpm):
    #Only use 30s clips of each track
    y_audio, sr = lb.load(path, duration=30)
    features = np.concatenate([
        root_mean_square(y_audio, sr),
        onset_envelope(y_audio, sr)
    ])
    return features, bpm


In [ ]:
CACHE_FILE = Path("features_cache.pkl")

if CACHE_FILE.exists():
    # Skips recomputation on subsequent runs
    print("Loading features from cache")
    X, y = joblib.load(CACHE_FILE)
    print(f"Loaded {len(X)} cached feature vectors.")
else:
    print("Extracting features")

    results = Parallel(n_jobs=-1)(
        delayed(process_track)(path, bpm)
        for path, bpm in tqdm(dataset, desc="Tracks")
    )
    X, y = zip(*results)

    joblib.dump((X, y), CACHE_FILE)
    print(f"Extracted features for {len(X)} tracks. Saved to {CACHE_FILE}.")

Loading features from cache
Loaded 664 cached feature vectors.
